In [8]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import sys

Инициализация SparkSession

In [9]:
spark_session = (
    SparkSession.builder
    .appName("ProgrammingLanguagesPopularityReport")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

spark_session.sparkContext.setLogLevel("WARN")

Загрузка и подготовка списка языков

In [3]:
from google.colab import files
uploaded = files.upload()

Saving programming-languages.csv to programming-languages.csv


In [6]:
from google.colab import files
uploaded = files.upload()

Saving posts_sample.xml to posts_sample.xml


In [15]:
langs_file_path = "/content/programming-languages.csv"
df_langs = spark_session.read.csv(langs_file_path, header=True)
lang_column_name = df_langs.columns[0]
valid_languages_set = {row[lang_column_name].lower() for row in df_langs.collect()}
broadcasted_langs = spark_session.sparkContext.broadcast(valid_languages_set)

print(f"Загружено и подготовлено {len(valid_languages_set)} названий языков программирования.")
df_langs.orderBy("name").show(10, truncate=False)

Загружено и подготовлено 698 названий языков программирования.
+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|@Formula  |https://en.wikipedia.org/wiki/Formula_language           |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
+----------+--

Чтение и первичная фильтрация данных из XML

In [19]:
posts_xml_path = "/content/posts_sample.xml"

raw_posts = spark_session.read.text(posts_xml_path).withColumnRenamed("value", "raw_xml")

candidate_posts = (
    raw_posts
    .where(F.col("raw_xml").contains('CreationDate="'))
    .where(F.col("raw_xml").contains('Tags="'))
)

posts_with_year_and_tags = (
    candidate_posts
    .select(
        F.regexp_extract("raw_xml", r'CreationDate="(\d{4})', 1).cast("int").alias("post_year"),
        F.regexp_extract("raw_xml", r'Tags="([^"]+)"', 1).alias("tag_sequence")
    )
    .where((F.col("post_year") >= 2010) & (F.col("post_year") <= 2020))
)

print(f"Количество постов после фильтрации: {posts_with_year_and_tags.count()}")
posts_with_year_and_tags.show(5, truncate=False)

Количество постов после фильтрации: 17642
+---------+--------------------------------------------------------------+
|post_year|tag_sequence                                                  |
+---------+--------------------------------------------------------------+
|2010     |&lt;c++&gt;&lt;character-encoding&gt;                         |
|2010     |&lt;sharepoint&gt;&lt;infopath&gt;                            |
|2010     |&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |
|2010     |&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;|
|2010     |&lt;java&gt;                                                  |
+---------+--------------------------------------------------------------+
only showing top 5 rows


Нормализация тегов


In [21]:
cleaned_tags = (
    posts_with_year_and_tags
    .withColumn("clean_tags", F.regexp_replace("tag_sequence", "&lt;", ""))
    .withColumn("clean_tags", F.regexp_replace("clean_tags", "&gt;", " "))
    .withColumn("tag_array", F.split(F.trim("clean_tags"), r"\s+"))
    .select(
        "post_year",  # ИСПРАВЛЕНО: было "year", стало "post_year"
        F.explode("tag_array").alias("raw_tag")
    )
    .withColumn("language_name", F.lower(F.trim("raw_tag")))
    .filter(F.col("language_name") != "")
    .select("post_year", "language_name") # ИСПРАВЛЕНО: было "year", стало "post_year"
)

cleaned_tags.show(10, truncate=False)

+---------+------------------+
|post_year|language_name     |
+---------+------------------+
|2010     |c++               |
|2010     |character-encoding|
|2010     |sharepoint        |
|2010     |infopath          |
|2010     |iphone            |
|2010     |app-store         |
|2010     |in-app-purchase   |
|2010     |symfony1          |
|2010     |schema            |
|2010     |doctrine          |
+---------+------------------+
only showing top 10 rows


Фильтрация по языкам

In [24]:
valid_langs_list = [(lang,) for lang in broadcasted_langs.value]
language_reference = spark_session.createDataFrame(valid_langs_list, ["language_name"])

joined_data = cleaned_tags.join(
    language_reference,
    on="language_name",
    how="inner"
)

language_stats = joined_data.groupBy("post_year", "language_name").count()
language_mentions = language_stats.withColumnRenamed("count", "mentions")

print("Топ упоминаний языков по годам:")
language_mentions.orderBy("post_year", F.desc("mentions")).show(20, truncate=False)

Топ упоминаний языков по годам:
+---------+-------------+--------+
|post_year|language_name|mentions|
+---------+-------------+--------+
|2010     |java         |52      |
|2010     |php          |46      |
|2010     |javascript   |44      |
|2010     |python       |26      |
|2010     |objective-c  |23      |
|2010     |c            |20      |
|2010     |ruby         |12      |
|2010     |delphi       |8       |
|2010     |perl         |3       |
|2010     |applescript  |3       |
|2010     |bash         |3       |
|2010     |r            |3       |
|2010     |haskell      |2       |
|2010     |f#           |2       |
|2010     |sed          |2       |
|2010     |actionscript |1       |
|2010     |racket       |1       |
|2010     |xpath        |1       |
|2010     |go           |1       |
|2010     |powershell   |1       |
+---------+-------------+--------+
only showing top 20 rows


Подсчет ранга внутри каждого года по убыванию упоминаний

In [25]:
yearly_window = Window.partitionBy("post_year").orderBy(
    F.desc("mentions"),
    F.asc("language_name")
)

top_10_per_year = (
    language_mentions
    .withColumn("position", F.row_number().over(yearly_window))
    .filter(F.col("position") <= 10)
    .select("post_year", "position", "language_name", "mentions")
    .orderBy("post_year", "position")
)

print("Топ-10 языков программирования по годам:")
top_10_per_year.show(120, truncate=False)

Топ-10 языков программирования по годам:
+---------+--------+-------------+--------+
|post_year|position|language_name|mentions|
+---------+--------+-------------+--------+
|2010     |1       |java         |52      |
|2010     |2       |php          |46      |
|2010     |3       |javascript   |44      |
|2010     |4       |python       |26      |
|2010     |5       |objective-c  |23      |
|2010     |6       |c            |20      |
|2010     |7       |ruby         |12      |
|2010     |8       |delphi       |8       |
|2010     |9       |applescript  |3       |
|2010     |10      |bash         |3       |
|2011     |1       |php          |102     |
|2011     |2       |java         |93      |
|2011     |3       |javascript   |83      |
|2011     |4       |python       |37      |
|2011     |5       |objective-c  |34      |
|2011     |6       |c            |24      |
|2011     |7       |ruby         |20      |
|2011     |8       |perl         |9       |
|2011     |9       |delphi       |8

Сохранение результата в Parquet

In [26]:
final_output_path = "report_top_languages_by_year"

top_10_per_year.write \
    .mode("overwrite") \
    .partitionBy("post_year") \
    .parquet(final_output_path)

print(f"Данные успешно сохранены в директорию: {final_output_path}")
print("Примечание: данные разбиты на партиции по столбцу 'post_year'.")

Данные успешно сохранены в директорию: report_top_languages_by_year
Примечание: данные разбиты на партиции по столбцу 'post_year'.


Формирование итогового отчета

In [28]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/report_top_languages_by_year",
    "zip",
    "/content/report_top_languages_by_year"
)
/
files.download("/content/report_top_languages_by_year.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
spark_session.stop()